# Market Evaluator — Pipeline Walkthrough

This notebook walks through the **product-space market evaluator** pipeline step by step.

It uses **pre-built sample outputs** so no API key is required. To run a live evaluation, see the cell at the bottom.

## Architecture recap

```
POST /evaluate
       │
       ▼
  run_pipeline()
       │
       ├─── Agent 1 (IncumbentsAgent)  ─┐
       ├─── Agent 2 (StartupsAgent)    ─┼─ asyncio.gather (concurrent)
       └─── Agent 3 (MarketScanAgent)  ─┘
                                        │
                                        ▼
                              Agent 4 (JudgementAgent)
                              pure maths — no API calls
                                        │
                                        ▼
                                  FinalResult
```

Each of Agents 1–3 follows the same pattern: **web search → clean → LLM extraction**.

In [ ]:
import sys
from pathlib import Path

# Make sure the project root is on sys.path
ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import os
# Set a dummy key so imports don't fail — replace with a real key to run live
os.environ.setdefault("OPENAI_API_KEY", "sk-demo")

print(f"Project root: {ROOT}")

## 1. Load a pre-built sample result

We ship three sample outputs in `demos/sample_outputs/`. Let's load the **AI sales automation** one.

In [ ]:
import json
from app.schemas import FinalResult

SAMPLE = Path(".").resolve() / "sample_outputs" / "ai-sales-automation.json"
result: FinalResult = FinalResult.model_validate_json(SAMPLE.read_text(encoding="utf-8"))

print(f"Product space : {result.product_space}")
print(f"Request ID    : {result.request_id}")
print(f"Errors        : {len(result.errors)}")

## 2. Inspect the judgement

In [ ]:
j = result.judgement
print(f"Verdict    : {j.verdict}")
print(f"Score      : {j.score}/10")
print(f"Confidence : {j.confidence}")
print()
print("Breakdown:")
print(f"  Growth score      : {j.breakdown.growth_score}")
print(f"  Competition score : {j.breakdown.competition_score}")
print(f"  White space       : {j.breakdown.white_space}")
print()
print("Summary:")
print(f"  {j.summary}")

## 3. Incumbent competitors

In [ ]:
import pandas as pd

players = result.incumbents.players if result.incumbents else []
df_incumbents = pd.DataFrame([
    {
        "Name": p.name,
        "Offerings": p.offerings[:60] + "..." if len(p.offerings) > 60 else p.offerings,
        "Target Customers": p.target_customers,
    }
    for p in players
])
df_incumbents

## 4. Startup funding landscape

In [ ]:
companies = result.startups.companies if result.startups else []
df_startups = pd.DataFrame([
    {
        "Name": c.name,
        "Stage": c.stage,
        "Raised ($M)": round(c.amount_usd / 1_000_000, 1) if c.amount_usd else None,
        "Date": str(c.date) if c.date else "—",
        "Lead Investors": ", ".join(c.lead_investors) if c.lead_investors else "—",
    }
    for c in companies
])
df_startups

In [ ]:
# Bar chart of funding by startup
ax = df_startups.set_index("Name")["Raised ($M)"].plot(
    kind="bar",
    title=f"Known Startup Funding — {result.product_space}",
    ylabel="USD (millions)",
    color="steelblue",
    rot=30,
)
ax.figure.tight_layout()

## 5. Market sizing

In [ ]:
ms = result.market_scan
if ms:
    print(f"TAM  : ${ms.tam_usd / 1e9:.1f}B ({ms.tam_year or 'n/a'})")
    if ms.sam_usd:
        print(f"SAM  : ${ms.sam_usd / 1e9:.1f}B ({ms.sam_year or 'n/a'})")
    print(f"CAGR : {ms.cagr_5y_percent:.1f}% (5-year)" if ms.cagr_5y_percent else "CAGR : —")
    print(f"Conf : {ms.confidence}")
    if ms.notes:
        print(f"Notes: {ms.notes}")

## 6. Score components breakdown (bar chart)

In [ ]:
import matplotlib.pyplot as plt

bd = result.judgement.breakdown
labels = ["Growth", "Competition", "White Space", "Final Score"]
values = [bd.growth_score, bd.competition_score, bd.white_space, result.judgement.score]
colors = ["#4CAF50" if v >= 6 else "#F44336" for v in values]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, values, color=colors, edgecolor="white", linewidth=0.5)
ax.axhline(6, color="gray", linestyle="--", linewidth=1, label="GO threshold (6)")
ax.set_ylim(0, 10)
ax.set_ylabel("Score (0–10)")
ax.set_title(f"Score Breakdown — {result.product_space}")
ax.legend()
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.15, str(val), ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.show()

## 7. Compare all three sample product spaces

In [ ]:
SAMPLES_DIR = Path(".").resolve() / "sample_outputs"

rows = []
for sample_file in sorted(SAMPLES_DIR.glob("*.json")):
    r = FinalResult.model_validate_json(sample_file.read_text(encoding="utf-8"))
    j = r.judgement
    ms = r.market_scan
    rows.append({
        "Product Space": r.product_space,
        "Verdict": j.verdict if j else "—",
        "Score": j.score if j else "—",
        "TAM ($B)": round(ms.tam_usd / 1e9, 1) if ms and ms.tam_usd else None,
        "CAGR (%)": ms.cagr_5y_percent if ms else None,
        "Incumbents": len(r.incumbents.players) if r.incumbents else 0,
        "Startups": r.startups.startup_count if r.startups else 0,
    })

pd.DataFrame(rows).set_index("Product Space")

## 8. (Optional) Run a live evaluation

Uncomment the cell below and set `OPENAI_API_KEY` to run the full pipeline. Expect **3–4 minutes** for the pipeline to complete.

> **Note:** The pipeline calls the OpenAI API for web search and structured extraction. Make sure your `.env` file has a valid `OPENAI_API_KEY`.

In [ ]:
# import nest_asyncio, asyncio
# from app.core.orchestrator import run_pipeline
#
# nest_asyncio.apply()  # allows asyncio.run() inside Jupyter
#
# PRODUCT_SPACE = "AI code review tools"
# result_live = asyncio.run(run_pipeline(PRODUCT_SPACE))
#
# j = result_live.judgement
# print(f"Verdict: {j.verdict}  Score: {j.score}/10")
# print(j.summary)